<a href="https://colab.research.google.com/github/bayanasar/membraneclaw/blob/main/experiments/notebooks/02_bellman_value_iteration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 — Bellman Equations, Value Iteration, Policy Extraction

Notebook 01 defined the MDP and left an open question: the "careful" policy
beats the "reckless" one, but is it *optimal*? Trying all 256 deterministic
policies by hand does not scale, and on a real problem it is hopeless.

This notebook answers the question exactly, using the two equations that
underpin nearly all of RL:

- the **Bellman expectation equation** — how good is *this* policy?
- the **Bellman optimality equation** — how good is the *best* policy?

Then it turns the second into an algorithm (**value iteration**) and recovers
the policy from the values (**policy extraction**).

The first cell re-states the environment from notebook 01 so this notebook
stands alone. Nothing in it is new — skim it and move on.

In [1]:
# --- environment from notebook 01, repeated so this notebook stands alone ---
from __future__ import annotations

from enum import IntEnum
from typing import NamedTuple

import numpy as np

class State(IntEnum):
    NO_INFO = 0           # nothing checked yet
    SALINITY_CHECKED = 1  # feed salinity known
    FOULING_CHECKED = 2   # fouling indicators known
    BOTH_CHECKED = 3      # both kinds of evidence in hand
    SUCCESS = 4           # problem solved (terminal)
    FAILURE = 5           # wrong or unsafe fix submitted (terminal)


class Action(IntEnum):
    CHECK_SALINITY = 0
    CHECK_FOULING = 1
    RUN_SIMULATION = 2
    SUBMIT_DIRECTLY = 3


N_STATES, N_ACTIONS = len(State), len(Action)
TERMINAL_STATES = frozenset({State.SUCCESS, State.FAILURE})
NONTERMINAL = [s for s in State if s not in TERMINAL_STATES]


def is_terminal(state) -> bool:
    return State(state) in TERMINAL_STATES


COST_CHECK = -0.5          # first look at a piece of evidence
COST_REPEAT_CHECK = -1.0   # re-checking something already known: pure waste
REWARD_SIM_SUCCESS = 9.0   # +10 outcome, minus the -1 implicit cost of simulating
REWARD_SIM_FAILURE = -11.0
REWARD_SUBMIT_SUCCESS = 10.0
REWARD_SUBMIT_FAILURE = -10.0

SIM_SUCCESS_PROB = {
    State.NO_INFO: 0.15,
    State.SALINITY_CHECKED: 0.55,
    State.FOULING_CHECKED: 0.45,
    State.BOTH_CHECKED: 0.95,
}
SUBMIT_SUCCESS_PROB = {
    State.NO_INFO: 0.05,
    State.SALINITY_CHECKED: 0.35,
    State.FOULING_CHECKED: 0.25,
    State.BOTH_CHECKED: 0.75,
}


class Transition(NamedTuple):
    prob: float
    next_state: State
    reward: float


# Evidence held in each non-terminal state, used to work out where a check lands.
_EVIDENCE = {
    State.NO_INFO: frozenset(),
    State.SALINITY_CHECKED: frozenset({Action.CHECK_SALINITY}),
    State.FOULING_CHECKED: frozenset({Action.CHECK_FOULING}),
    State.BOTH_CHECKED: frozenset({Action.CHECK_SALINITY, Action.CHECK_FOULING}),
}
_STATE_BY_EVIDENCE = {ev: st for st, ev in _EVIDENCE.items()}


def transitions(state, action) -> tuple[Transition, ...]:
    # Every outcome of taking `action` in `state`, probabilities summing to 1.
    state, action = State(state), Action(action)

    if is_terminal(state):
        return (Transition(1.0, state, 0.0),)

    if action in (Action.CHECK_SALINITY, Action.CHECK_FOULING):
        already_known = action in _EVIDENCE[state]
        next_state = (
            state if already_known
            else _STATE_BY_EVIDENCE[_EVIDENCE[state] | {action}]
        )
        reward = COST_REPEAT_CHECK if already_known else COST_CHECK
        return (Transition(1.0, next_state, reward),)

    if action is Action.RUN_SIMULATION:
        p = SIM_SUCCESS_PROB[state]
        return (
            Transition(p, State.SUCCESS, REWARD_SIM_SUCCESS),
            Transition(1.0 - p, State.FAILURE, REWARD_SIM_FAILURE),
        )

    p = SUBMIT_SUCCESS_PROB[state]
    return (
        Transition(p, State.SUCCESS, REWARD_SUBMIT_SUCCESS),
        Transition(1.0 - p, State.FAILURE, REWARD_SUBMIT_FAILURE),
    )

def transition_tables() -> tuple[np.ndarray, np.ndarray]:
    # Dense tables for exact methods: P[s, a, s'] and expected R[s, a].
    P = np.zeros((N_STATES, N_ACTIONS, N_STATES))
    R = np.zeros((N_STATES, N_ACTIONS))
    for s in State:
        for a in Action:
            for prob, next_state, reward in transitions(s, a):
                P[s, a, next_state] += prob
                R[s, a] += prob * reward
    return P, R


P, R = transition_tables()

def policy_matrices(pi, P, R):
    # Collapse an MDP + policy into an MRP: (P_pi [S,S], R_pi [S]).
    P_pi = np.einsum("sa,sat->st", pi, P)
    R_pi = np.einsum("sa,sa->s", pi, R)
    return P_pi, R_pi


def deterministic(action_of_state) -> np.ndarray:
    # Build a [S, A] policy matrix from a {state: action} mapping.
    pi = np.zeros((N_STATES, N_ACTIONS))
    for s in State:
        pi[s, action_of_state(s)] = 1.0
    return pi


def step(state, action, rng) -> tuple[State, float, bool]:
    # Sample one environment step: (next_state, reward, done).
    outcomes = transitions(state, action)
    probs = [t.prob for t in outcomes]
    chosen = outcomes[rng.choice(len(outcomes), p=probs)]
    return chosen.next_state, chosen.reward, is_terminal(chosen.next_state)


def rollout(policy_fn, rng, max_steps=20):
    # Run one episode; return a list of (state, action, reward) triples.
    s, traj = State.NO_INFO, []
    for _ in range(max_steps):
        a = policy_fn(s, rng)
        ns, r, done = step(s, a, rng)
        traj.append((s, a, r))
        s = ns
        if done:
            break
    return traj


def show(traj):
    total = sum(r for _, _, r in traj)
    for s, a, r in traj:
        print(f"  {State(s).name:<18} --{Action(a).name:<16}--> {r:+.1f}")
    print(f"  total (undiscounted) = {total:+.1f}")
    return total

print("environment ready:", N_STATES, "states,", N_ACTIONS, "actions")

environment ready: 6 states, 4 actions


## Returns and the discount factor

The agent maximises the **return** — cumulative discounted reward from time $t$:

$$G_t = R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+3} + \cdots
      = \sum_{k=0}^{\infty} \gamma^k R_{t+k+1}$$

$\gamma$ controls how much the future counts. At $\gamma = 0$ the agent is
myopic and grabs the best immediate reward; at $\gamma \to 1$ it weighs distant
rewards nearly as heavily as immediate ones.

$\gamma$ matters here in a specific way: **gathering evidence is an investment**.
A check pays `-0.5` now for a better payoff later. Discount the future hard
enough and that investment stops being worth it — a point we can locate exactly
later in this notebook.

We use $\gamma = 0.95$. Since $\gamma < 1$ and rewards are bounded, every value
below is a finite number.

In [2]:
GAMMA = 0.95

## The value functions

Two objects, both defined relative to a policy $\pi$:

$$v_\pi(s) = \mathbb{E}_\pi[G_t \mid S_t = s] \qquad\text{(state-value)}$$
$$q_\pi(s, a) = \mathbb{E}_\pi[G_t \mid S_t = s, A_t = a] \qquad\text{(action-value)}$$

$v_\pi(s)$ answers "starting here and following $\pi$, what do I collect?"
$q_\pi(s,a)$ answers "starting here, taking $a$ *once*, then following $\pi$?"

That one-step deviation in $q$ is the whole engine of policy improvement. If
some action beats what $\pi$ currently does, $\pi$ can be improved by switching
to it — which is exactly what policy extraction does at the end of this
notebook.

## The Bellman expectation equation

Values obey a recursion. Split the return into the first reward and the rest:

$$v_\pi(s) = \sum_a \pi(a \mid s) \Big[ R(s,a) + \gamma \sum_{s'} P(s' \mid s,a)\, v_\pi(s') \Big]$$

In the MRP form from notebook 01 ($P_\pi$, $R_\pi$), this is compact:

$$v_\pi = R_\pi + \gamma P_\pi v_\pi$$

This is a **linear system**, and rearranging gives a closed-form solve:

$$(I - \gamma P_\pi)\, v_\pi = R_\pi \quad \Longrightarrow \quad v_\pi = (I - \gamma P_\pi)^{-1} R_\pi$$

$I - \gamma P_\pi$ is invertible whenever $\gamma < 1$, so one `np.linalg.solve`
gives the exact value of any policy.

In [3]:
def evaluate_exact(pi, gamma=GAMMA):
    # Solve (I - gamma P_pi) v = R_pi directly for the exact v_pi.
    P_pi, R_pi = policy_matrices(pi, P, R)
    return np.linalg.solve(np.eye(N_STATES) - gamma * P_pi, R_pi)


careful_pi = deterministic(lambda s: {
    State.NO_INFO: Action.CHECK_SALINITY,
    State.SALINITY_CHECKED: Action.CHECK_FOULING,
    State.FOULING_CHECKED: Action.CHECK_SALINITY,
    State.BOTH_CHECKED: Action.RUN_SIMULATION,
}.get(s, Action.RUN_SIMULATION))

reckless_pi = deterministic(lambda s: Action.SUBMIT_DIRECTLY)

v_careful = evaluate_exact(careful_pi)
v_reckless = evaluate_exact(reckless_pi)

print(f"{'state':<18}{'v_careful':>12}{'v_reckless':>12}")
for s in State:
    print(f"{s.name:<18}{v_careful[s]:>12.3f}{v_reckless[s]:>12.3f}")

state                v_careful  v_reckless
NO_INFO                  6.245      -9.000
SALINITY_CHECKED         7.100      -3.000
FOULING_CHECKED          7.100      -5.000
BOTH_CHECKED             8.000       5.000
SUCCESS                  0.000       0.000
FAILURE                  0.000       0.000


`v_careful[NO_INFO] = 6.245` is the exact expected discounted return of the
careful policy from the start state — no sampling, no error bars. Compare it to
the Monte Carlo average in notebook 01 (`+6.96`, undiscounted): the gap is
$\gamma$ shrinking the delayed `+9`, not estimation error.

The reckless policy's `-9.0` at `NO_INFO` is its single immediate submission,
undiscounted because the episode ends right there.

Terminal states have value exactly `0`, as they must: they absorb and pay
nothing.

### Iterative policy evaluation

The direct solve is $O(|\mathcal{S}|^3)$ and needs the full model. The
alternative is to treat the Bellman equation as an **update rule** and apply it
repeatedly:

$$v_{k+1}(s) \leftarrow \sum_a \pi(a\mid s)\Big[R(s,a) + \gamma \sum_{s'} P(s'\mid s,a)\, v_k(s')\Big]$$

This converges to $v_\pi$ from any starting point, because the Bellman operator
is a **$\gamma$-contraction**: each sweep shrinks the worst-case error by a
factor of $\gamma$. That contraction property is why nearly every method in RL
converges at all, and we can watch it happen.

In [4]:
def evaluate_iterative(pi, gamma=GAMMA, tol=1e-12, max_sweeps=500):
    # Repeatedly apply the Bellman expectation backup. Returns (v, history).
    P_pi, R_pi = policy_matrices(pi, P, R)
    v = np.zeros(N_STATES)
    history = [v.copy()]
    for _ in range(max_sweeps):
        v_new = R_pi + gamma * P_pi @ v
        history.append(v_new.copy())
        if np.max(np.abs(v_new - v)) < tol:
            v = v_new
            break
        v = v_new
    return v, np.array(history)


v_iter, hist = evaluate_iterative(careful_pi)
print(f"converged in {len(hist) - 1} sweeps")
print("matches the direct solve:", np.allclose(v_iter, v_careful))

print(f"\n{'sweep':>6}" + "".join(f"{s.name[:8]:>10}" for s in NONTERMINAL))
for k in [0, 1, 2, 3, 5, 10, 20, len(hist) - 1]:
    if k < len(hist):
        print(f"{k:>6}" + "".join(f"{hist[k][s]:>10.3f}" for s in NONTERMINAL))

converged in 4 sweeps
matches the direct solve: True

 sweep   NO_INFO  SALINITY  FOULING_  BOTH_CHE
     0     0.000     0.000     0.000     0.000
     1    -0.500    -0.500    -0.500     8.000
     2    -0.975     7.100     7.100     8.000
     3     6.245     7.100     7.100     8.000
     4     6.245     7.100     7.100     8.000


Watch the first few sweeps. After sweep 1 every state holds only its immediate
reward — the checks show `-0.5`, `BOTH_CHECKED` shows `+8.0`. Information about
the eventual payoff then propagates **backwards** one step per sweep: sweep 2
reaches the states one check away, sweep 3 the states two checks away. This
backward flow of value is the mechanism behind every dynamic-programming method
here.

In [5]:
# The contraction in action: error against the true v_pi, per sweep.
err = np.max(np.abs(hist - v_careful), axis=1)
print(f"{'sweep':>6}{'max error':>14}{'ratio':>9}   (gamma = 0.95)")
for k in range(1, len(err)):
    ratio = err[k] / err[k - 1] if err[k - 1] > 0 else float("nan")
    print(f"{k:>6}{err[k]:>14.6f}{ratio:>9.3f}")

 sweep     max error    ratio   (gamma = 0.95)
     1      7.600000    0.950
     2      7.220000    0.950
     3      0.000000    0.000
     4      0.000000      nan


The error hits **exactly zero at sweep 3**, which deserves an explanation,
because the contraction bound only promises a factor-of-$\gamma$ shrink per
sweep — geometric decay, never exact termination.

Both statements are true. The contraction is a worst-case *guarantee*, and this
MDP beats it. Under the careful policy the state graph is a short **acyclic
chain**: `NO_INFO -> SALINITY_CHECKED -> BOTH_CHECKED -> terminal`. Each sweep
propagates exact value back one link, so after as many sweeps as the chain is
long, every state has its true value and there is nothing left to shrink.

Geometric decay is what you see when values must flow around **cycles**, so that
each state's value depends on its own — the usual case. Add a cycle here and the
`0.95` ratio appears:

In [6]:
# A policy with a cycle: keep re-checking what you already know.
stalling_pi = deterministic(lambda s: Action.CHECK_SALINITY)
v_stall, hist_stall = evaluate_iterative(stalling_pi)
err_stall = np.max(np.abs(hist_stall - v_stall), axis=1)

print("stalling policy (self-loops at SALINITY_CHECKED -> cyclic):")
print(f"{'sweep':>6}{'max error':>14}{'ratio':>9}")
for k in range(1, 11):
    ratio = err_stall[k] / err_stall[k - 1] if err_stall[k - 1] > 0 else float("nan")
    print(f"{k:>6}{err_stall[k]:>14.6f}{ratio:>9.3f}")

print(f"\nsweeps to converge: {len(hist_stall) - 1}")
print(f"v_stall(NO_INFO) = {v_stall[State.NO_INFO]:.3f}  (endless re-checking, never commits)")

stalling policy (self-loops at SALINITY_CHECKED -> cyclic):
 sweep     max error    ratio
     1     19.000000    0.950
     2     18.050000    0.950
     3     17.147500    0.950
     4     16.290125    0.950
     5     15.475619    0.950
     6     14.701838    0.950
     7     13.966746    0.950
     8     13.268409    0.950
     9     12.604988    0.950
    10     11.974739    0.950

sweeps to converge: 500
v_stall(NO_INFO) = -19.500  (endless re-checking, never commits)


There it is: the ratio pins to `0.95`, exactly $\gamma$, and convergence takes
hundreds of sweeps instead of three. The stalling policy loops forever paying
`-1.0` per wasted re-check, so its value is a large negative number that the
backups have to grind toward geometrically.

This also prices $\gamma$ for you: at `0.999` you would need roughly 200x more
sweeps for the same accuracy, since $\log(\epsilon)/\log(\gamma)$ governs the
sweep count.

## The Bellman optimality equation

Evaluation scores a policy you hand it. To find the *best* policy, replace the
average over $\pi$ with a **maximum** over actions:

$$v_*(s) = \max_a \Big[ R(s,a) + \gamma \sum_{s'} P(s' \mid s,a)\, v_*(s') \Big]$$

$$q_*(s,a) = R(s,a) + \gamma \sum_{s'} P(s' \mid s,a) \max_{a'} q_*(s', a')$$

The `max` makes this **non-linear**, so there is no matrix inverse this time.
But the operator is still a $\gamma$-contraction, so iterating it still
converges — and that iteration is the algorithm.

## Value iteration

Apply the optimality backup until values stop moving:

1. start with $v_0 = 0$
2. $v_{k+1}(s) \leftarrow \max_a \big[R(s,a) + \gamma \sum_{s'} P(s'\mid s,a) v_k(s')\big]$
3. stop when $\max_s |v_{k+1}(s) - v_k(s)| < \theta$

Note what is *absent*: no policy is stored or improved. Value iteration works
purely in value space, and the policy is read off at the very end.

In [7]:
def value_iteration(gamma=GAMMA, tol=1e-12, max_sweeps=1000):
    # Iterate the Bellman optimality backup. Returns (v_star, q_star, history).
    v = np.zeros(N_STATES)
    history = [v.copy()]
    for _ in range(max_sweeps):
        q = R + gamma * P @ v          # [S, A] one-step lookahead
        v_new = q.max(axis=1)
        history.append(v_new.copy())
        if np.max(np.abs(v_new - v)) < tol:
            v = v_new
            break
        v = v_new
    q = R + gamma * P @ v
    return v, q, np.array(history)


v_star, q_star, vi_hist = value_iteration()
print(f"value iteration converged in {len(vi_hist) - 1} sweeps\n")
print(f"{'state':<18}{'v*(s)':>10}{'v_careful':>12}{'gain':>9}")
for s in State:
    print(f"{s.name:<18}{v_star[s]:>10.3f}{v_careful[s]:>12.3f}{v_star[s] - v_careful[s]:>9.3f}")

value iteration converged in 4 sweeps

state                  v*(s)   v_careful     gain
NO_INFO                6.245       6.245    0.000
SALINITY_CHECKED       7.100       7.100    0.000
FOULING_CHECKED        7.100       7.100    0.000
BOTH_CHECKED           8.000       8.000    0.000
SUCCESS                0.000       0.000    0.000
FAILURE                0.000       0.000    0.000


The gain column is **zero in every state**. The hand-written careful policy was
optimal all along.

That is a satisfying result but a slightly anticlimactic one, so it is worth
being clear about what was actually gained here. Before value iteration we had a
policy that beat its rival in a sampled comparison; we had no way to know
whether some untried policy beat it in turn. Now we have a *certificate*: $v_*$
is the best achievable value in every state, `careful` attains it, and no policy
— among the 256 deterministic ones or the infinitely many stochastic ones — does
better. Confirming optimality is as much a result as discovering it.

### Watching value iteration converge

In [8]:
print(f"{'sweep':>6}" + "".join(f"{s.name[:8]:>10}" for s in NONTERMINAL) + "   greedy action at NO_INFO")
for k in range(min(9, len(vi_hist))):
    row = "".join(f"{vi_hist[k][s]:>10.3f}" for s in NONTERMINAL)
    q_k = R + GAMMA * P @ vi_hist[k]
    print(f"{k:>6}{row}   {Action(q_k[State.NO_INFO].argmax()).name}")

 sweep   NO_INFO  SALINITY  FOULING_  BOTH_CHE   greedy action at NO_INFO
     0     0.000     0.000     0.000     0.000   CHECK_SALINITY
     1    -0.500     0.000    -0.500     8.000   CHECK_SALINITY
     2    -0.500     7.100     7.100     8.000   CHECK_SALINITY
     3     6.245     7.100     7.100     8.000   CHECK_SALINITY
     4     6.245     7.100     7.100     8.000   CHECK_SALINITY


Two things to read off this table.

**The values change; the greedy action does not.** `CHECK_SALINITY` is already
greedy at sweep 0, when every value estimate is still zero, and it stays greedy
throughout. In this MDP the immediate rewards alone are enough to rank the
actions correctly at `NO_INFO`: a check costs `-0.5`, while committing blind has
an *immediate* expected reward around `-8`. There is no lookahead needed to see
that gathering evidence is better here.

That is a property of this problem, not a general rule. In MDPs where the
short-term and long-term rankings disagree — the classic case being a costly
action that unlocks a large delayed payoff — the greedy action does flip
mid-run, and reading a policy out before convergence gives you a wrong one that
looks settled. The safe habit is the same either way: **extract the policy only
at the fixed point.**

**Value still propagates backwards one link per sweep.** Sweep 1 knows only
immediate rewards; sweep 2 has pushed `BOTH_CHECKED`'s `+8` back into the
one-check states; sweep 3 reaches `NO_INFO`. That backward flow is the mechanism
every dynamic-programming method shares.

In [9]:
# The optimal action-value table: q*(s, a).
print("q*(s, a)  -  best action per row marked with *")
print(f"{'':<18}" + "".join(f"{a.name:>17}" for a in Action))
for s in NONTERMINAL:
    best = q_star[s].argmax()
    cells = "".join(
        f"{q_star[s, a]:>16.3f}" + ("*" if a == best else " ") for a in Action
    )
    print(f"{s.name:<18}{cells}")

q*(s, a)  -  best action per row marked with *
                     CHECK_SALINITY    CHECK_FOULING   RUN_SIMULATION  SUBMIT_DIRECTLY
NO_INFO                      6.245*           6.245           -8.000           -9.000 
SALINITY_CHECKED             5.745            7.100*           0.000           -3.000 
FOULING_CHECKED              7.100*           5.745           -2.000           -5.000 
BOTH_CHECKED                 6.600            6.600            8.000*           5.000 


## Policy extraction

Value iteration returns numbers, not behaviour. To get a policy, act **greedily**
with respect to $q_*$:

$$\pi_*(s) = \arg\max_a q_*(s, a)
           = \arg\max_a \Big[R(s,a) + \gamma \sum_{s'} P(s'\mid s,a)\, v_*(s')\Big]$$

The greedy policy w.r.t. the *optimal* values is optimal — that is the guarantee
that makes the two-stage "solve for values, then extract" approach valid.

In [10]:
def extract_policy(q):
    # Greedy (deterministic) policy from an action-value table.
    pi = np.zeros((N_STATES, N_ACTIONS))
    pi[np.arange(N_STATES), q.argmax(axis=1)] = 1.0
    return pi


optimal_pi = extract_policy(q_star)

print("optimal policy:")
for s in NONTERMINAL:
    a = Action(optimal_pi[s].argmax())
    print(f"  {s.name:<18} -> {a.name:<16}  q* = {q_star[s].max():.3f}")

# Verify: evaluating the extracted policy must reproduce v*.
v_check = evaluate_exact(optimal_pi)
print("\nevaluating the extracted policy reproduces v*:", np.allclose(v_check, v_star))

optimal policy:
  NO_INFO            -> CHECK_SALINITY    q* = 6.245
  SALINITY_CHECKED   -> CHECK_FOULING     q* = 7.100
  FOULING_CHECKED    -> CHECK_SALINITY    q* = 7.100
  BOTH_CHECKED       -> RUN_SIMULATION    q* = 8.000

evaluating the extracted policy reproduces v*: True


### Reading the answer

The optimal policy is **check both pieces of evidence, then run the simulation**
— exactly the structure of the hand-written "careful" policy.

The `q*` table above is where the reasoning lives. Read the `NO_INFO` row:
committing right away is worth `-8.0` (simulate) or `-9.0` (submit), while
either check is worth `+6.245`. A `-0.5` check that buys access to a `+8.0`
state is not a close call. Then read the `BOTH_CHECKED` row: with full evidence,
simulating (`+8.0`) beats submitting blind (`+5.0`), so the simulator's implicit
`-1` cost is repaid by the jump from 75% to 95% success.

Note also the tie at `NO_INFO`: `CHECK_SALINITY` and `CHECK_FOULING` are both
worth `6.245`. Order does not matter, only that both get done — the two routes
to `BOTH_CHECKED` cost the same. `argmax` breaks the tie arbitrarily, which is
worth remembering when a policy looks oddly specific about something that does
not matter.

In [11]:
print(f"{'state':<18}{'careful':<18}{'optimal':<18}{'agree'}")
for s in NONTERMINAL:
    ca, oa = Action(careful_pi[s].argmax()), Action(optimal_pi[s].argmax())
    print(f"{s.name:<18}{ca.name:<18}{oa.name:<18}{'yes' if ca == oa else 'NO'}")

print("\nvalue of each action at NO_INFO under q*:")
for a in Action:
    print(f"  {a.name:<18}{q_star[State.NO_INFO, a]:>8.3f}")

state             careful           optimal           agree
NO_INFO           CHECK_SALINITY    CHECK_SALINITY    yes
SALINITY_CHECKED  CHECK_FOULING     CHECK_FOULING     yes
FOULING_CHECKED   CHECK_SALINITY    CHECK_SALINITY    yes
BOTH_CHECKED      RUN_SIMULATION    RUN_SIMULATION    yes

value of each action at NO_INFO under q*:
  CHECK_SALINITY       6.245
  CHECK_FOULING        6.245
  RUN_SIMULATION      -8.000
  SUBMIT_DIRECTLY     -9.000


Every row agrees. The intuition that built `careful` by hand was correct, and now
it is proven rather than assumed — that is the difference computation buys.

## How much does $\gamma$ matter?

Earlier I suggested that discounting hard enough should make evidence-gathering
worthless, since a check pays `-0.5` now for a benefit one step later. That is a
testable claim, so let us test it rather than assert it: sweep $\gamma$ from 0 to
0.99 and watch the optimal first action.

In [12]:
print(f"{'gamma':>7}{'v*(NO_INFO)':>14}   optimal first action")
for g in [0.0, 0.1, 0.2, 0.3, 0.5, 0.7, 0.9, 0.95, 0.99]:
    v_g, q_g, _ = value_iteration(gamma=g)
    print(f"{g:>7.2f}{v_g[State.NO_INFO]:>14.3f}   {Action(q_g[State.NO_INFO].argmax()).name}")

  gamma   v*(NO_INFO)   optimal first action
   0.00        -0.500   CHECK_SALINITY
   0.10        -0.470   CHECK_SALINITY
   0.20        -0.280   CHECK_SALINITY
   0.30         0.070   CHECK_SALINITY
   0.50         1.250   CHECK_SALINITY
   0.70         3.070   CHECK_SALINITY
   0.90         5.530   CHECK_SALINITY
   0.95         6.245   CHECK_SALINITY
   0.99         6.846   CHECK_SALINITY


**No flip.** `CHECK_SALINITY` stays optimal all the way down to $\gamma = 0$,
where the agent is perfectly myopic and cares only about the next reward.

My prediction was wrong, and the `q*` table explains why. A myopic agent
compares immediate rewards only: `-0.5` for a check against roughly `-8` for
committing with no evidence. Checking wins on the immediate term alone — no
patience required. The `-0.5` never needed the future to justify it; it was
simply the cheapest available action.

What $\gamma$ *does* change is the **value**, which climbs from `-0.5` to
`+6.85`: a myopic agent walks the same path but sees almost none of the reward
waiting at the end of it.

Is the policy here *ever* sensitive to $\gamma$? We can settle that by sweeping
the check cost and $\gamma$ together, instead of speculating.

In [13]:
# Does any check cost make the optimal first action depend on gamma?
gammas = [0.0, 0.2, 0.4, 0.6, 0.8, 0.9, 0.95, 0.99]
print(f"{'COST_CHECK':>11}  " + "".join(f"{g:>7}" for g in gammas))

original_cost = COST_CHECK
for cost in [-0.5, -4.0, -7.0, -7.75, -8.0, -8.25, -9.0]:
    COST_CHECK = cost
    P_c, R_c = transition_tables()          # rebuild tables with the new cost
    acts = []
    for g in gammas:
        v_c = np.zeros(N_STATES)
        for _ in range(5000):
            q_c = R_c + g * P_c @ v_c
            nv = q_c.max(axis=1)
            if np.max(np.abs(nv - v_c)) < 1e-13:
                v_c = nv
                break
            v_c = nv
        q_c = R_c + g * P_c @ v_c
        acts.append(Action(q_c[State.NO_INFO].argmax()).name[:5])
    flip = "  <- depends on gamma" if len(set(acts)) > 1 else ""
    print(f"{cost:>11.2f}  " + "".join(f"{a:>7}" for a in acts) + flip)

COST_CHECK = original_cost                  # restore
P, R = transition_tables()

 COST_CHECK      0.0    0.2    0.4    0.6    0.8    0.9   0.95   0.99
      -0.50    CHECK  CHECK  CHECK  CHECK  CHECK  CHECK  CHECK  CHECK
      -4.00    CHECK  CHECK  CHECK  CHECK  CHECK  CHECK  CHECK  CHECK
      -7.00    CHECK  CHECK  CHECK  CHECK  CHECK  CHECK  CHECK  CHECK
      -7.75    CHECK  CHECK  CHECK  CHECK  CHECK  CHECK  CHECK  CHECK
      -8.00    CHECK  CHECK  CHECK  CHECK  CHECK  CHECK  CHECK  CHECK
      -8.25    RUN_S  RUN_S  RUN_S  RUN_S  RUN_S  RUN_S  RUN_S  RUN_S
      -9.00    RUN_S  RUN_S  RUN_S  RUN_S  RUN_S  RUN_S  RUN_S  RUN_S


Every row is constant across $\gamma$. The behaviour switches from `CHECK` to
`RUN_S` between a cost of `-8.00` and `-8.25` — and that boundary is the *same*
whether the agent is perfectly myopic or nearly perfectly patient.

The reason is visible in the `q*` table from earlier: `RUN_SIMULATION` at
`NO_INFO` is worth `-8.0`, and because it terminates the episode immediately,
that value contains no discounted future at all. So the comparison at `NO_INFO`
is between a check's cost and a fixed `-8.0`, and $\gamma$ multiplies terms on
only one side of it — never enough to reorder them here.

So in this MDP, **$\gamma$ moves the values but never the policy**. That is a
real structural property, and it is the sort of thing you find by sweeping
rather than by reasoning from the textbook description of discounting. In MDPs
where a costly action unlocks a large *delayed* payoff, $\gamma$ absolutely does
flip the optimal policy — the standard intuition is sound, it just does not bind
here.

The durable lesson: $\gamma$ is a **modelling decision** rather than a tuning
knob, and whether it changes behaviour depends on whether short- and long-run
rankings genuinely conflict in your problem. Check; do not assume.

## What we can now do — and what we cannot

Value iteration gave us the exact optimum in a handful of sweeps. But look at
what it required: **full access to $P$ and $R$**. Every backup sums over all
next states, weighted by exact probabilities.

Real problems rarely offer that:

- the dynamics are unknown (you have a plant, not its transition matrix);
- or the state space is far too large to sweep;
- or the model exists but is wrong in ways you cannot audit.

So the rest of the series drops the model. Notebook 03 estimates values from
**sampled episodes alone**, using nothing but `step()`.

| Notebook | Adds |
| --- | --- |
| 01 | states, actions, transitions, rewards, terminal states |
| **02 — this one** | Bellman equations, value iteration, policy extraction |
| **03** | Monte Carlo policy evaluation — values from samples |
| **04** | REINFORCE, baselines, and a critic |
| **05** | PPO's probability ratio and clipping |

Keep `v_star` and `q_star` in mind — from here on they are the **ground truth**
every sampled method gets graded against.

In [14]:
# Ground truth carried forward to notebooks 03-05.
print("v*  =", np.array2string(v_star, precision=4, suppress_small=True))
print("optimal:", {State(s).name: Action(optimal_pi[s].argmax()).name for s in NONTERMINAL})

v*  = [6.245 7.1   7.1   8.    0.    0.   ]
optimal: {'NO_INFO': 'CHECK_SALINITY', 'SALINITY_CHECKED': 'CHECK_FOULING', 'FOULING_CHECKED': 'CHECK_SALINITY', 'BOTH_CHECKED': 'RUN_SIMULATION'}
